## Object Detection and Image Segmentation

This notebook walks through the key concepts from today's session using standard datasets and pretrained models from `torchvision`. No custom training is required — the focus is on understanding how these models work and how to use them.

**Topics covered:**
1. Object Detection with Faster R-CNN
2. Intersection over Union (IoU)
3. Non-Maximum Suppression (NMS)
4. Semantic Segmentation with DeepLabV3
5. A brief look at Vision Transformers (ViT)

**Dataset:** We use the COCO-pretrained weights that ship with `torchvision`, applied to a few sample images downloaded from open sources.

---
## 0. Setup

In [ ]:
# Standard installs — run once if any package is missing
# !pip install torch torchvision matplotlib requests Pillow --quiet

In [1]:
import torch
import torchvision
import torchvision.transforms as T
import torchvision.ops as ops
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.10.0+cpu
Torchvision: 0.25.0+cpu
Device: cpu


In [2]:
# Utility: load an image from a URL
def load_image_from_url(url):
    response = requests.get(url, timeout=10)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    return image

# Utility: display one or more images side by side
def show_images(images, titles=None, figsize=(14, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i], fontsize=12)
    plt.tight_layout()
    plt.show()

---
## 1. Object Detection with Faster R-CNN

Faster R-CNN is a two-stage detector:
- **Stage 1 — Region Proposal Network (RPN):** Scans the feature map and proposes candidate regions likely to contain objects.
- **Stage 2 — ROI Head:** Takes each proposed region and classifies it + refines the bounding box.

We load a model pretrained on COCO (80 object classes) and run inference on a sample image.

In [ ]:
# COCO class names (80 classes)
COCO_CLASSES = [
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "train", "truck", "boat", "traffic light", "fire hydrant",
    "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse",
    "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork",
    "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush"
]

# Assign a distinct color to each class for visualization
np.random.seed(42)
CLASS_COLORS = np.random.randint(100, 255, size=(len(COCO_CLASSES), 3)).tolist()

In [ ]:
# Load a pretrained Faster R-CNN model
# Weights: COCO_V1 — trained on MS COCO with 80 categories
weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
detection_model = fasterrcnn_resnet50_fpn(weights=weights)
detection_model.eval()

print("Model loaded. Number of parameters:",
      sum(p.numel() for p in detection_model.parameters()) // 1_000_000, "M")

In [ ]:
# Load a sample street scene image
# We use a Creative Commons image from Wikimedia
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/640px-Camponotus_flavomarginatus_ant.jpg"

# Fallback: use a local test image from torchvision if URL fails
try:
    sample_image = load_image_from_url(
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/38/Shopping_Center_Los_Cerritos_Center.jpg/640px-Shopping_Center_Los_Cerritos_Center.jpg"
    )
    print("Image loaded from URL. Size:", sample_image.size)
except Exception:
    # Generate a simple synthetic image if network is unavailable
    sample_image = Image.fromarray(
        np.random.randint(100, 200, (480, 640, 3), dtype=np.uint8)
    )
    print("Using synthetic fallback image.")

show_images([sample_image], titles=["Input Image"])

In [ ]:
def run_detection(model, image, confidence_threshold=0.5):
    """
    Run Faster R-CNN on a PIL image.
    Returns boxes, labels, and scores above the confidence threshold.
    """
    transform = T.ToTensor()
    img_tensor = transform(image).unsqueeze(0)  # shape: [1, C, H, W]

    with torch.no_grad():
        predictions = model(img_tensor)

    pred = predictions[0]
    keep = pred["scores"] >= confidence_threshold

    boxes  = pred["boxes"][keep].numpy()
    labels = pred["labels"][keep].numpy()
    scores = pred["scores"][keep].numpy()

    return boxes, labels, scores


def draw_detections(image, boxes, labels, scores):
    """
    Draw bounding boxes and labels on a copy of the image.
    """
    fig, ax = plt.subplots(1, figsize=(12, 7))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        color = [c / 255.0 for c in CLASS_COLORS[label]]
        class_name = COCO_CLASSES[label] if label < len(COCO_CLASSES) else str(label)

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 5,
            f"{class_name}: {score:.2f}",
            color="white", fontsize=9, fontweight="bold",
            bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor="none")
        )

    ax.set_title(f"{len(boxes)} object(s) detected", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Run detection
boxes, labels, scores = run_detection(detection_model, sample_image, confidence_threshold=0.5)

print(f"Detections above 0.5 confidence: {len(boxes)}")
for i, (label, score) in enumerate(zip(labels, scores)):
    class_name = COCO_CLASSES[label] if label < len(COCO_CLASSES) else str(label)
    print(f"  [{i+1}] {class_name:20s}  score: {score:.3f}")

In [ ]:
# Visualize detections
draw_detections(sample_image, boxes, labels, scores)

**What to observe:**
- Each box has a class name and confidence score.
- Lowering the threshold (e.g., 0.3) will show more detections — but also more false positives.
- Try changing `confidence_threshold` in `run_detection()` and re-running.

---
## 2. Intersection over Union (IoU)

IoU measures how well a predicted bounding box overlaps with the ground truth.

```
IoU = Area of Intersection / Area of Union
```

Range: 0 (no overlap) to 1 (perfect overlap). Typical threshold: 0.5.

In [ ]:
def compute_iou(box_a, box_b):
    """
    Compute IoU between two boxes in [x1, y1, x2, y2] format.
    """
    # Intersection coordinates
    ix1 = max(box_a[0], box_b[0])
    iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2])
    iy2 = min(box_a[3], box_b[3])

    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def visualize_iou(ground_truth, prediction, label_gt="Ground Truth", label_pred="Prediction"):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect("equal")

    # Ground truth box
    gt = patches.Rectangle(
        (ground_truth[0], ground_truth[1]),
        ground_truth[2] - ground_truth[0],
        ground_truth[3] - ground_truth[1],
        linewidth=2, edgecolor="steelblue", facecolor="steelblue", alpha=0.3,
        label=label_gt
    )
    # Prediction box
    pred = patches.Rectangle(
        (prediction[0], prediction[1]),
        prediction[2] - prediction[0],
        prediction[3] - prediction[1],
        linewidth=2, edgecolor="tomato", facecolor="tomato", alpha=0.3,
        label=label_pred
    )

    ax.add_patch(gt)
    ax.add_patch(pred)

    iou = compute_iou(ground_truth, prediction)
    ax.set_title(f"IoU = {iou:.3f}", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return iou

In [ ]:
# Three IoU examples: poor, acceptable, and good overlap
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

examples = [
    {"gt": [1, 1, 5, 5], "pred": [4, 4, 8, 8], "title": "Poor overlap"},
    {"gt": [1, 1, 6, 6], "pred": [2, 2, 7, 7], "title": "Moderate overlap"},
    {"gt": [1, 1, 6, 6], "pred": [1.5, 1.5, 5.5, 5.5], "title": "Good overlap"},
]

for ax, ex in zip(axes, examples):
    gt, pred = ex["gt"], ex["pred"]
    iou = compute_iou(gt, pred)

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect("equal")

    gt_rect = patches.Rectangle(
        (gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1],
        linewidth=2, edgecolor="steelblue", facecolor="steelblue", alpha=0.4, label="Ground Truth"
    )
    pred_rect = patches.Rectangle(
        (pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
        linewidth=2, edgecolor="tomato", facecolor="tomato", alpha=0.4, label="Prediction"
    )
    ax.add_patch(gt_rect)
    ax.add_patch(pred_rect)
    ax.set_title(f"{ex['title']}\nIoU = {iou:.3f}", fontsize=11)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

plt.suptitle("IoU Examples", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Use torchvision's built-in box_iou for batch computation
# This is what detection pipelines use internally

boxes_a = torch.tensor([[1.0, 1.0, 5.0, 5.0],
                         [2.0, 2.0, 8.0, 8.0]])

boxes_b = torch.tensor([[4.0, 4.0, 8.0, 8.0],
                         [3.0, 3.0, 7.0, 7.0]])

iou_matrix = ops.box_iou(boxes_a, boxes_b)
print("IoU matrix (2x2 — each row is a box_a, each column is a box_b):")
print(iou_matrix.numpy().round(3))
print()
print("Entry [0, 0] = IoU between box_a[0] and box_b[0]:", iou_matrix[0, 0].item())

---
## 3. Non-Maximum Suppression (NMS)

When a model detects an object, it typically produces several overlapping boxes. NMS keeps only the most confident box per object and suppresses the rest.

**Algorithm:**
1. Sort boxes by confidence score (high to low).
2. Keep the top box.
3. Remove any box that overlaps with it by more than the IoU threshold.
4. Repeat from step 2 with the remaining boxes.

In [ ]:
# Simulate what detection looks like BEFORE NMS
# Multiple overlapping boxes for the same two objects

raw_boxes = torch.tensor([
    [50,  50, 200, 200],   # object 1 — best box
    [55,  55, 205, 205],   # object 1 — slight shift
    [60,  52, 195, 198],   # object 1 — another overlap
    [70,  65, 210, 210],   # object 1 — weaker overlap
    [250, 80, 400, 230],   # object 2 — best box
    [255, 82, 405, 232],   # object 2 — slight shift
    [260, 78, 398, 228],   # object 2 — another overlap
], dtype=torch.float)

raw_scores = torch.tensor([0.95, 0.88, 0.76, 0.61, 0.92, 0.85, 0.70])

# Apply NMS
iou_threshold = 0.5
kept_indices = ops.nms(raw_boxes, raw_scores, iou_threshold)

print(f"Boxes before NMS: {len(raw_boxes)}")
print(f"Boxes after  NMS: {len(kept_indices)}")
print(f"Kept indices:     {kept_indices.tolist()}")
print(f"Kept scores:      {raw_scores[kept_indices].tolist()}")

In [ ]:
def plot_nms_comparison(raw_boxes, raw_scores, kept_indices, canvas_size=(460, 310)):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    titles = ["Before NMS — all proposals", "After NMS — kept boxes"]
    kept_set = set(kept_indices.tolist())

    for col, ax in enumerate(axes):
        ax.set_xlim(0, canvas_size[0])
        ax.set_ylim(canvas_size[1], 0)  # y-axis flipped (image coordinates)
        ax.set_facecolor("#f5f5f5")
        ax.set_title(titles[col], fontsize=12)
        ax.grid(False)

        for i, (box, score) in enumerate(zip(raw_boxes.tolist(), raw_scores.tolist())):
            x1, y1, x2, y2 = box
            if col == 1 and i not in kept_set:
                continue
            color = "steelblue" if i in kept_set else "lightcoral"
            alpha = 0.8 if i in kept_set else 0.35
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor=color, alpha=alpha
            )
            ax.add_patch(rect)
            ax.text(x1 + 3, y1 + 15, f"{score:.2f}",
                    fontsize=8, color="white", fontweight="bold")

    plt.tight_layout()
    plt.show()

plot_nms_comparison(raw_boxes, raw_scores, kept_indices)

**Key point:** NMS is applied *after* the model produces all proposals. The IoU threshold controls how aggressively overlapping boxes are suppressed. A lower threshold keeps fewer boxes.

---
## 4. Image Segmentation with DeepLabV3

Semantic segmentation assigns a class label to every pixel in the image. DeepLabV3 uses **atrous (dilated) convolutions** to capture multi-scale context without losing spatial resolution.

The model we load here is also pretrained on COCO, with 21 classes (Pascal VOC categories).

In [ ]:
# Pascal VOC class names and a fixed color palette for visualization
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "dining table", "dog",
    "horse", "motorbike", "person", "potted plant", "sheep",
    "sofa", "train", "tv/monitor"
]

# Colormap: each class gets a unique color
VOC_COLORMAP = np.array([
    [  0,   0,   0], [128,   0,   0], [  0, 128,   0], [128, 128,   0],
    [  0,   0, 128], [128,   0, 128], [  0, 128, 128], [128, 128, 128],
    [ 64,   0,   0], [192,   0,   0], [ 64, 128,   0], [192, 128,   0],
    [ 64,   0, 128], [192,   0, 128], [ 64, 128, 128], [192, 128, 128],
    [  0,  64,   0], [128,  64,   0], [  0, 192,   0], [128, 192,   0],
    [  0,  64, 128]
], dtype=np.uint8)

In [ ]:
# Load pretrained DeepLabV3
seg_weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
seg_model = deeplabv3_resnet50(weights=seg_weights)
seg_model.eval()

print("Segmentation model loaded.")
print("Number of output classes:", 21)

In [ ]:
def run_segmentation(model, image):
    """
    Run DeepLabV3 on a PIL image.
    Returns a 2D numpy array where each value is a class index.
    """
    preprocess = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])
    input_tensor = preprocess(image).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)["out"][0]  # shape: [21, H, W]

    seg_map = output.argmax(dim=0).numpy()  # shape: [H, W]
    return seg_map


def colorize_segmentation(seg_map, colormap):
    """Convert a class-index map to an RGB image using the colormap."""
    h, w = seg_map.shape
    color_image = np.zeros((h, w, 3), dtype=np.uint8)
    for class_idx in range(len(colormap)):
        color_image[seg_map == class_idx] = colormap[class_idx]
    return color_image

In [ ]:
# Use the same sample image from Section 1, or load a new one
seg_image = sample_image.copy()

# Run segmentation
seg_map = run_segmentation(seg_model, seg_image)
seg_color = colorize_segmentation(seg_map, VOC_COLORMAP)

# Overlay: blend original image with segmentation map
original_np = np.array(seg_image.resize((seg_map.shape[1], seg_map.shape[0])))
overlay = (0.55 * original_np + 0.45 * seg_color).astype(np.uint8)

# Show results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(seg_image)
axes[0].set_title("Original Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(seg_color)
axes[1].set_title("Segmentation Map", fontsize=12)
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Overlay (original + mask)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Inspect which classes were detected and how much of the image they cover
unique_classes, pixel_counts = np.unique(seg_map, return_counts=True)
total_pixels = seg_map.size

print(f"{'Class':<20} {'Pixels':>8}  {'Coverage':>10}")
print("-" * 42)
for cls_idx, count in sorted(zip(unique_classes, pixel_counts), key=lambda x: -x[1]):
    cls_name = VOC_CLASSES[cls_idx] if cls_idx < len(VOC_CLASSES) else str(cls_idx)
    pct = 100.0 * count / total_pixels
    print(f"{cls_name:<20} {count:>8,}   {pct:>8.2f}%")

In [ ]:
# Visualize the class distribution as a bar chart
class_names = [VOC_CLASSES[i] if i < len(VOC_CLASSES) else str(i) for i in unique_classes]
coverage = [100.0 * c / total_pixels for c in pixel_counts]

# Sort by coverage
sorted_pairs = sorted(zip(class_names, coverage), key=lambda x: -x[1])
names_sorted, cov_sorted = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names_sorted, cov_sorted, color="steelblue", edgecolor="white", width=0.6)
ax.set_ylabel("Coverage (%)", fontsize=11)
ax.set_title("Pixel Coverage per Detected Class", fontsize=13)
ax.set_xticklabels(names_sorted, rotation=30, ha="right", fontsize=10)
ax.grid(axis="y", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## 5. Detection vs. Segmentation — Side-by-Side

It is useful to see both outputs on the same image to understand what each task gives you.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Detection (left) ---
axes[0].imshow(sample_image)
for box, label, score in zip(boxes, labels, scores):
    x1, y1, x2, y2 = box
    color = [c / 255.0 for c in CLASS_COLORS[label]]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor=color, facecolor="none"
    )
    axes[0].add_patch(rect)
    axes[0].text(
        x1, y1 - 5,
        f"{COCO_CLASSES[label]}: {score:.2f}",
        color="white", fontsize=8, fontweight="bold",
        bbox=dict(facecolor=color, alpha=0.75, pad=2, edgecolor="none")
    )
axes[0].set_title("Object Detection\n(bounding boxes + class labels)", fontsize=12)
axes[0].axis("off")

# --- Segmentation (right) ---
axes[1].imshow(overlay)
axes[1].set_title("Semantic Segmentation\n(pixel-level class labels)", fontsize=12)
axes[1].axis("off")

plt.suptitle("Detection gives bounding boxes. Segmentation gives pixel masks.",
             fontsize=11, y=1.01, style="italic")
plt.tight_layout()
plt.show()

---
## 6. Vision Transformer (ViT) — Patch Visualization

ViT divides an image into fixed-size patches and treats each patch as a token — similar to how words are tokens in NLP. Here we visualize this patch decomposition to build intuition, then run a classification with a pretrained ViT.

In [ ]:
def visualize_patches(image, patch_size=16):
    """
    Resize the image to 224x224 and draw the patch grid that ViT sees.
    """
    img_resized = image.resize((224, 224))
    img_np = np.array(img_resized)

    num_patches_per_side = 224 // patch_size
    n_patches = num_patches_per_side ** 2

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: original with grid overlay
    axes[0].imshow(img_np)
    for i in range(0, 224, patch_size):
        axes[0].axhline(i, color="white", linewidth=0.5, alpha=0.6)
        axes[0].axvline(i, color="white", linewidth=0.5, alpha=0.6)
    axes[0].set_title(
        f"ViT input: 224x224 image\ndivided into {n_patches} patches ({patch_size}x{patch_size} each)",
        fontsize=11
    )
    axes[0].axis("off")

    # Right: show a grid of individual patches (first 64)
    n_show = min(64, n_patches)
    grid_size = int(np.ceil(np.sqrt(n_show)))
    patch_canvas = np.ones((grid_size * (patch_size + 2), grid_size * (patch_size + 2), 3),
                           dtype=np.uint8) * 50

    for idx in range(n_show):
        row = idx // grid_size
        col = idx % grid_size
        patch_row = (idx // num_patches_per_side) * patch_size
        patch_col = (idx % num_patches_per_side) * patch_size
        patch = img_np[patch_row:patch_row + patch_size,
                       patch_col:patch_col + patch_size]
        y_start = row * (patch_size + 2)
        x_start = col * (patch_size + 2)
        patch_canvas[y_start:y_start + patch_size,
                     x_start:x_start + patch_size] = patch

    axes[1].imshow(patch_canvas)
    axes[1].set_title(f"First {n_show} individual patches", fontsize=11)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
    print(f"Patch size: {patch_size}x{patch_size} pixels")
    print(f"Total patches per image: {n_patches}")
    print(f"Each patch is flattened to a vector of size: {patch_size*patch_size*3} = {patch_size}^2 x 3 (RGB)")

visualize_patches(sample_image, patch_size=16)

In [ ]:
# Run inference with a pretrained ViT from torchvision
from torchvision.models import vit_b_16, ViT_B_16_Weights

vit_weights = ViT_B_16_Weights.IMAGENET1K_V1
vit_model = vit_b_16(weights=vit_weights)
vit_model.eval()

print("ViT-Base/16 loaded.")
print("Number of parameters:",
      sum(p.numel() for p in vit_model.parameters()) // 1_000_000, "M")

In [ ]:
def classify_with_vit(model, weights, image, top_k=5):
    """
    Classify an image using a pretrained ViT model.
    Returns the top-k predicted classes and their probabilities.
    """
    preprocess = weights.transforms()
    input_tensor = preprocess(image).unsqueeze(0)

    with torch.no_grad():
        logits = model(input_tensor)

    probs = torch.softmax(logits, dim=1)[0]
    top_probs, top_indices = probs.topk(top_k)

    categories = weights.meta["categories"]
    results = [
        (categories[idx.item()], prob.item())
        for idx, prob in zip(top_indices, top_probs)
    ]
    return results


predictions = classify_with_vit(vit_model, vit_weights, sample_image, top_k=5)

print("ViT-Base/16 — Top-5 Predictions")
print("-" * 38)
for rank, (class_name, prob) in enumerate(predictions, 1):
    bar = "|" * int(prob * 40)
    print(f"  {rank}. {class_name:<25} {prob*100:5.1f}%  {bar}")

In [ ]:
# Compare ViT to a CNN on the same image
from torchvision.models import resnet50, ResNet50_Weights

cnn_weights = ResNet50_Weights.IMAGENET1K_V2
cnn_model = resnet50(weights=cnn_weights)
cnn_model.eval()

def classify_with_cnn(model, weights, image, top_k=5):
    preprocess = weights.transforms()
    input_tensor = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    top_probs, top_indices = probs.topk(top_k)
    categories = weights.meta["categories"]
    return [(categories[idx.item()], prob.item()) for idx, prob in zip(top_indices, top_probs)]

cnn_preds = classify_with_cnn(cnn_model, cnn_weights, sample_image, top_k=5)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, title, preds in zip(axes, ["ViT-Base/16", "ResNet-50"], [predictions, cnn_preds]):
    names  = [p[0][:30] for p in preds]
    confs  = [p[1] * 100 for p in preds]
    colors = ["steelblue" if i == 0 else "lightsteelblue" for i in range(len(preds))]
    ax.barh(names[::-1], confs[::-1], color=colors[::-1], edgecolor="white")
    ax.set_xlabel("Confidence (%)")
    ax.set_title(title, fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Top-5 Predictions: ViT vs CNN on the same image",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Summary and Exercises

**What we covered:**

| Topic | Model Used | Key Output |
|---|---|---|
| Object Detection | Faster R-CNN | Bounding boxes + class labels |
| IoU | Manual + `torchvision.ops` | Overlap score (0–1) |
| NMS | `torchvision.ops.nms` | Filtered box set |
| Semantic Segmentation | DeepLabV3 | Per-pixel class map |
| Classification with ViT | ViT-Base/16 | Class probabilities |

---

**Exercises (try on your own):**

1. Change the `confidence_threshold` in Section 1 to `0.3` and observe what changes. How many additional detections appear?

2. In Section 3, modify `iou_threshold` in `ops.nms()` — try `0.3` and `0.7`. What happens at each extreme?

3. In Section 4, load a different image (e.g., an outdoor scene) and run segmentation. Which classes appear in the pixel coverage chart?

4. In Section 6, try `patch_size=32` and compare the number of patches. How does patch size affect what the model "sees"?

5. Apply both detection and segmentation to the same new image and discuss: which output would be more useful for counting objects? Which for measuring object area?

In [ ]:
# Scratch cell — use this for your exercises
